In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
from tqdm import tqdm
import numpy as np
from scipy.spatial import ConvexHull
from tqdm import trange


In [ ]:

ECG_DIR="data/ukb_heartmesh_processed/ecg_resting"
mesh_root = 'data/ukb_heartmesh_processed/ctrl_mesh'
def segment_volumes(pts):
    """Convex-hull volume per frame
    pts : (T, N, 3)    -> returns (T,) volumes."""
    vols = np.zeros(pts.shape[0], dtype=np.float32)
    for t in range(pts.shape[0]):
        try:
            vols[t] = ConvexHull(pts[t]).volume
        except:
            vols[t] = np.nan
    return vols


In [ ]:
from datetime import datetime
print(datetime.now())
all_ecg_delay_mat = dict(np.load('ukb_heart_mesh_visual/all_ecg_delay_matrix.npz'))

print(datetime.now())

In [ ]:

def get_ed_time(meshpath):
    if meshpath is None: return None
    mesh_pts = np.load(meshpath)
    return segment_volumes(mesh_pts)

        
meshpaths = [f'{mesh_root}/{fn[:-4]}_30099_2_0/frames.npy' for fn in all_ecg_delay_mat['filename'].tolist()]
meshpaths = [p if os.path.exists(p) else None for p in meshpaths]
results = [None] * len(meshpaths)

with ProcessPoolExecutor(max_workers=40) as executor:
    future_to_idx = {executor.submit(get_ed_time, fn): i for i, fn in enumerate(meshpaths)}
    for fut in tqdm(as_completed(future_to_idx, timeout=300), total=len(meshpaths)):
        i = future_to_idx[fut]
        try:
            results[i] = fut.result()
        except Exception as e:
            print(f"Task {i} failed: {e}")
    

In [ ]:
mesh_frame_num = np.array([len(results[i]) if results[i] is not None else 0 for i in range(len(results))])
np.bincount(mesh_frame_num)

In [ ]:
volume_changes = [results[i] for i in np.where(mesh_frame_num!=0)[0]]
volume_changes = np.stack(volume_changes)
volumes_clip = np.clip(volume_changes, np.nanpercentile(volume_changes, 5), np.nanpercentile(volume_changes, 95))
volumes = np.asarray(volumes_clip, dtype=float)
baseline = volumes[:, [0]]
volumes_pct = (volumes - baseline) / baseline * 100
volumes_pct_mean = np.abs(np.array([v.mean() for v in volumes_pct]).round()).astype(int)
valid_volume_mask = (volumes_pct_mean > 1) & (volumes_pct_mean < 31) & ((volumes_pct < 31).all(1))
volumes_pct = volumes_pct[valid_volume_mask]
np.bincount(volumes_pct_mean[valid_volume_mask])

In [ ]:

def plot_cohort_trajectory(volumes, time_points=np.arange(0, 32*50, 32), sample_n=300):
    """
    volumes: array (N, T), normalized volume change per subject
    time_points: array (T,)
    """
    N, T = volumes.shape
    rng = np.random.default_rng(0)
    idx = rng.choice(N, size=min(sample_n, N), replace=False)

    med = np.nanmedian(volumes, axis=0)
    q1 = np.nanpercentile(volumes, 25, axis=0)
    q3 = np.nanpercentile(volumes, 75, axis=0)

    plt.figure(figsize=(3, 2))

    for i in idx:
        plt.plot(time_points, volumes[i], color="gray", alpha=0.1, linewidth=0.7)

    plt.plot(time_points, med, color="black", linewidth=2.5, label="Median")
    plt.fill_between(time_points, q1, q3, color="black", alpha=0.15, label="Interquartile Range")

    plt.xlabel("")
    plt.ylabel("")
    # plt.title("Cohort volume change trajectory")
    plt.legend(loc='upper center',)
    plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

group1 = [v.argmin()<=16 for v in volume_changes[valid_volume_mask]]
group2 = [v.argmin()>20 for v in volume_changes[valid_volume_mask]]
print(sum(group1), sum(group2))
plot_cohort_trajectory(volumes_pct[group1])
plt.savefig('heart_mesh_volume_change_less16.svg')
plot_cohort_trajectory(volumes_pct[group2])
plt.savefig('heart_mesh_volume_change_larger20.svg')

In [ ]:
physctrl_pred_root = 'checkpoints/ecg_heart_8L_256d_49frames/vis_eval'
pred_fn = [f'{physctrl_pred_root}/{fn}' for fn in os.listdir(physctrl_pred_root) if fn.endswith('_pred.npy')]

pred_volumes = [None] * len(pred_fn)

with ProcessPoolExecutor(max_workers=40) as executor:
    future_to_idx = {executor.submit(get_ed_time, fn): i for i, fn in enumerate(pred_fn)}
    for fut in tqdm(as_completed(future_to_idx, timeout=300), total=len(pred_fn)):
        i = future_to_idx[fut]
        try:
            pred_volumes[i] = fut.result()
        except Exception as e:
            print(f"Task {i} failed: {e}")

In [ ]:
mesh_frame_num = np.array([len(pred_volumes[i]) if pred_volumes[i] is not None else 0 for i in range(len(pred_volumes))])
np.bincount(mesh_frame_num)

In [ ]:
volume_changes = [pred_volumes[i] for i in np.where(mesh_frame_num!=0)[0]]
volume_changes = np.stack(volume_changes)
volumes_clip = np.clip(volume_changes, np.nanpercentile(volume_changes, 5), np.nanpercentile(volume_changes, 95))
volumes = np.asarray(volumes_clip, dtype=float)
baseline = volumes[:, [0]]
volumes_pct = (volumes - baseline) / baseline * 100
volumes_pct_mean = np.abs(np.array([v.mean() for v in volumes_pct]).round()).astype(int)
valid_volume_mask = (volumes_pct_mean > 1) & (volumes_pct_mean < 31) & ((volumes_pct < 31).all(1))
volumes_pct = volumes_pct[valid_volume_mask]
np.bincount(volumes_pct_mean[valid_volume_mask])

plot_cohort_trajectory(volumes_pct, time_points=np.arange(32, 32*50, 32),)
plt.savefig('heart_mesh_volume_change_physctrl.svg')

In [ ]:
get_ed_time(f'{physctrl_pred_root}/{pred_fn[0]}')

In [ ]:


ecg_df = {'T_ED': [], 'QRS_delay_mat': [], 'Tpeak_delay_mat': [], 'Toffset_delay_mat': []}
for i in trange(len(results)):
    if results[i] is None: continue
    ecg_df['T_ED'].append(results[i].argmin())
    ecg_df['QRS_delay_mat'].append(all_ecg_delay_mat['QRS'][i])
    ecg_df['Tpeak_delay_mat'].append(all_ecg_delay_mat['T_peak'][i])
    ecg_df['Toffset_delay_mat'].append(all_ecg_delay_mat['T_offset'][i])

In [ ]:
ecg_df['QRS_delay_mat'] = np.stack(ecg_df['QRS_delay_mat'])

In [ ]:
import seaborn as sns
sns.histplot(ecg_df['T_ED'])


In [ ]:
import umap


In [ ]:

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,      # increase for more global structure (try 30–50)
    min_dist=0.1,        # smaller = tighter clusters
    metric='euclidean',
    random_state=42,
    verbose=True,
    low_memory=False     # faster if you have RAM
)
qrs = ecg_df['QRS_delay_mat'].reshape(len(ecg_df['QRS_delay_mat']), -1)
qrs_embedding = reducer.fit_transform(qrs)  # shape: (45000, 2)

In [ ]:
qrs.shape, qrs_embedding.shape

In [ ]:
np.bincount(ecg_df['T_ED'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
plt.figure(figsize=(15,15))
plot_df = pd.DataFrame({
    'UMAP1': qrs_embedding[:, 0],
    'UMAP2': qrs_embedding[:, 1],
    'EndSystole': np.array(ecg_df['T_ED']),
})
plot_df["EndSystole_bin"] = pd.qcut(plot_df["EndSystole"], q=5, duplicates="drop")
sns.scatterplot(data=plot_df, x='UMAP1', y='UMAP2', hue='EndSystole_bin', s=10, alpha=0.9, palette='tab10')


In [ ]:
import numpy as np
from scipy.stats import ttest_ind
from itertools import combinations

def cohens_d(x1, x2):
    """
    x1, x2: arrays of shape (n_samples, 12, 12)
    returns: (12, 12) Cohen's d
    """
    n1, n2 = x1.shape[0], x2.shape[0]

    m1 = np.nanmean(x1, axis=0)
    m2 = np.nanmean(x2, axis=0)

    s1 = np.nanstd(x1, axis=0, ddof=1)
    s2 = np.nanstd(x2, axis=0, ddof=1)

    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    d = (m1 - m2) / pooled_sd
    return d

def pairwise_cell_ttests(mats, groups, equal_var=False):
    """
    mats:   array of shape (N, 12, 12)
    groups: array of shape (N,)
    """
    mats = np.asarray(mats)
    groups = np.asarray(groups)

    unique_groups = np.sort(np.unique(groups))
    pairs = list(combinations(unique_groups, 2))

    # Store p-values and t-statistics:
    # shape = (num_pairs, 12, 12)
    t_stats = np.zeros((len(pairs), mats.shape[1], mats.shape[2]))
    p_vals = np.ones((len(pairs), mats.shape[1], mats.shape[2]))
    n_sample, diff = [], []
    for k, (g1, g2) in enumerate(pairs):
        x1 = mats[groups == g1]  # (n1, 12, 12)
        x2 = mats[groups == g2]  # (n2, 12, 12)

        # t-test along axis=0 compares each cell across samples
        res = ttest_ind(x1, x2, axis=0, equal_var=equal_var, nan_policy='omit')
        n_sample.append((len(x1), len(x2)))
        t_stats[k] = res.statistic
        p_vals[k] = res.pvalue
        diff.append(cohens_d(x1, x2))

    return pairs, t_stats, p_vals, np.array(n_sample), np.array(diff)

In [ ]:
plot_df["EndSystole_bin"] = pd.qcut(plot_df["EndSystole"], q=8, duplicates="drop")

In [ ]:
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['QRS_delay_mat'], np.array(plot_df["EndSystole_bin"].astype("category").cat.codes))


In [ ]:
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['QRS_delay_mat'], np.array(pd.qcut(plot_df["EndSystole"], q=2, duplicates="drop").astype("category").cat.codes))
plt.figure()
sns.heatmap(p_vals[0], cmap='cividis_r')
plt.title(f'p-value, n={n_samples[0]}')
plt.figure()
sns.heatmap(diffs[0], cmap='coolwarm')
plt.title(f"Cohen's d, n={n_samples[0]}")

In [ ]:
print("Groups:")
print(plot_df["EndSystole_bin"].astype("category").cat.categories.to_numpy())

In [ ]:
ngroup = len(plot_df["EndSystole_bin"].astype("category").cat.categories.to_numpy())

fig, axes = plt.subplots(ngroup, ngroup-1, figsize=(16, 20), sharex=True, sharey=True)
# axes = np.array(axes)#.ravel()

for i, (pair, p_mat, n_sample, diff) in enumerate(zip(pairs, p_vals, n_samples, diffs)):
    # mask = p_mat >= 0.05
    # print(p_mat[~np.isnan(p_mat)].min(), p_mat[~np.isnan(p_mat)].max())
    sns.heatmap(
        p_mat,
        # mask=mask,
        ax=axes[pair[0], pair[1]-1],
        cmap="cividis_r",
        vmax=1,
        vmin=0,
        cbar=False,
    )
    axes[pair[0], pair[1]-1].set_title(f"p-value, n={n_sample[0]} vs n={n_sample[1]}")

    sns.heatmap(
        diff,
        # mask=mask,
        ax=axes[pair[1], pair[0]],
        cmap="coolwarm",
        center=0,
        vmax=0.1,
        vmin=-0.1,
        cbar=False,
        square=True
    )
    axes[pair[1], pair[0]].set_title(f"Cohen's d, n={n_sample[0]} vs n={n_sample[1]}")

# hide unused axes
# for j in range(len(axes)):
#     axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
len(pairs)

In [ ]:

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,      # increase for more global structure (try 30–50)
    min_dist=0.1,        # smaller = tighter clusters
    metric='euclidean',
    verbose=True,
    low_memory=False     # faster if you have RAM
)
tpeak = np.stack(ecg_df['Tpeak_delay_mat']).reshape(len(ecg_df['Tpeak_delay_mat']), -1)
tpeak_embedding = reducer.fit_transform(tpeak)  # shape: (45000, 2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
plt.figure(figsize=(15,15))
plot_df = pd.DataFrame({
    'UMAP1': tpeak_embedding[:, 0],
    'UMAP2': tpeak_embedding[:, 1],
    'EndSystole': np.array(ecg_df['T_ED']),
})
plot_df["EndSystole_bin"] = pd.qcut(plot_df["EndSystole"], q=5, duplicates="drop")
sns.scatterplot(data=plot_df, x='UMAP1', y='UMAP2', hue='EndSystole_bin', s=10, alpha=0.9, palette='tab10')


In [ ]:
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['Tpeak_delay_mat'], np.array(pd.qcut(plot_df["EndSystole"], q=2, duplicates="drop").astype("category").cat.codes))
print(np.array(pd.qcut(plot_df["EndSystole"], q=2, duplicates="drop").astype("category").cat.categories))
plt.figure()
sns.heatmap(p_vals[0], cmap='cividis_r')
plt.title(f'p-value, n={n_samples[0]}')
plt.figure()
sns.heatmap(diffs[0], cmap='coolwarm')
plt.title(f"Cohen's d, n={n_samples[0]}")

In [ ]:
group_bin = pd.qcut(plot_df["EndSystole"], q=8, duplicates="drop")
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['Tpeak_delay_mat'], np.array(group_bin.astype("category").cat.codes))

ngroup = len(group_bin.astype("category").cat.categories.to_numpy())
print(group_bin.astype("category").cat.categories)
fig, axes = plt.subplots(ngroup, ngroup-1, figsize=(16, 20), sharex=True, sharey=True)
# axes = np.array(axes)#.ravel()

for i, (pair, p_mat, n_sample, diff) in enumerate(zip(pairs, p_vals, n_samples, diffs)):
    # mask = p_mat >= 0.05
    # print(p_mat[~np.isnan(p_mat)].min(), p_mat[~np.isnan(p_mat)].max())
    sns.heatmap(
        p_mat,
        # mask=mask,
        ax=axes[pair[0], pair[1]-1],
        cmap="cividis_r",
        vmax=1,
        vmin=0,
        cbar=False,
    )
    axes[pair[0], pair[1]-1].set_title(f"p-value, n={n_sample[0]} vs n={n_sample[1]}")

    sns.heatmap(
        diff,
        # mask=mask,
        ax=axes[pair[1], pair[0]],
        cmap="coolwarm",
        center=0,
        # vmax=0.5,
        # vmin=-0.5,
        cbar=True,
        square=True
    )
    axes[pair[1], pair[0]].set_title(f"Cohen's d, n={n_sample[0]} vs n={n_sample[1]}")

# hide unused axes
# for j in range(len(axes)):
#     axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_combined_heatmap(pvals, dvals, labels=None, alpha=0.05):
    n = pvals.shape[0]
    assert pvals.shape == dvals.shape

    # Base matrix just for display
    base = np.zeros((n, n))
    np.fill_diagonal(base, np.nan)

    fig, ax = plt.subplots(figsize=(4, 4))

    # --- Upper triangle: p-values ---
    mask_upper = np.tril(np.ones_like(pvals, dtype=bool))  # hide lower + diagonal
    sns.heatmap(
        pvals,
        mask=mask_upper,
        cmap="viridis_r",
        vmin=0,
        vmax=1,
        square=True,
        cbar=False,
        # cbar_kws={"label": "p-value"},
        ax=ax,
        annot=False
    )

    # --- Lower triangle: Cohen's d ---
    mask_lower = np.triu(np.ones_like(dvals, dtype=bool))  # hide upper + diagonal
    sns.heatmap(
        dvals,
        mask=mask_lower,
        cmap="coolwarm",
        center=0,
        square=True,
        cbar=False,
        ax=ax,
        annot=False
    )

    # Optional: add text annotations
    for i in range(n):
        for j in range(n):
            if i < j:
                ax.text(j + 0.5, i + 0.5, f"{pvals[i, j]:.3f}",
                        ha="center", va="center", fontsize=7, color="black")
            elif i > j:
                ax.text(j + 0.5, i + 0.5, f"{dvals[i, j]:.2f}",
                        ha="center", va="center", fontsize=7, color="black")

    # Labels
    if labels is not None:
        ax.set_xticks(np.arange(n) + 0.5)
        ax.set_yticks(np.arange(n) + 0.5)
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticklabels(labels, rotation=0)

    ax.set_title("Upper triangle: $p$-values | Lower triangle: Cohen's $d$")

    plt.tight_layout()
    # plt.show()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def p_to_annot(p):
    if np.isnan(p):
        return ""
    if p < 1e-4:
        return "***"
    elif p < 1e-3:
        return "**"
    elif p < 1e-2:
        return "*"
    else:
        return f"{p:.2f}"

def plot_combined_heatmap_with_star(p_mat, d_mat, labels=None, cmap_p="YlGnBu", cmap_d="coolwarm"):
    """
    p_mat: 2D array of p-values
    d_mat: 2D array of Cohen's d
    labels: optional tick labels
    """
    p_mat = np.asarray(p_mat)
    d_mat = np.asarray(d_mat)
    n = p_mat.shape[0]

    # Build combined display matrix
    combined = np.full((n, n), np.nan, dtype=float)
    upper = np.triu_indices(n, k=1)
    lower = np.tril_indices(n, k=-1)

    # For color: upper triangle uses p-values, lower triangle uses d-values
    # We'll normalize separately by plotting two masked heatmaps on top of each other.
    # First create annotation matrix
    annot = np.empty((n, n), dtype=object)
    annot[:] = ""

    for i, j in zip(*upper):
        annot[i, j] = p_to_annot(p_mat[i, j])

    for i, j in zip(*lower):
        annot[i, j] = f"{d_mat[i, j]:.2f}"

    fig, ax = plt.subplots(figsize=(4, 4))

    # Lower triangle: Cohen's d
    mask_upper = np.triu(np.ones_like(d_mat, dtype=bool), k=0)
    sns.heatmap(
        d_mat,
        mask=mask_upper,
        cmap=cmap_d,
        center=0,
        annot=annot,
        fmt="",
        square=True,
        # linewidths=0.5,
        # linecolor="white",
        cbar=False,
        ax=ax
    )

    # Upper triangle: p-values
    mask_lower = np.tril(np.ones_like(p_mat, dtype=bool), k=0)
    sns.heatmap(
        p_mat,
        mask=mask_lower,
        cmap=cmap_p,
        vmin=0,
        vmax=1,
        annot=annot,
        fmt="",
        square=True,
        # linewidths=0.5,
        # linecolor="white",
        cbar=False,
        ax=ax
    )

    if labels is not None:
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticklabels(labels, rotation=0)

    ax.set_title("")
    plt.tight_layout()
    # plt.show()

In [ ]:
import xml.etree.ElementTree as ET

def get_ecg_signal(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    # Find the MedianSamples section
    median_samples = root.find('.//MedianSamples')
    if median_samples is None:
        raise ValueError("MedianSamples not found in XML")
    
    # Get resolution
    resolution = float(median_samples.find('Resolution').text)
    
    # Initialize dictionary for signals
    signals = []
    leads = []
    # Iterate over WaveformData elements
    for waveform in median_samples.findall('WaveformData'):
        lead = waveform.get('lead')
        leads.append(lead)
        data_str = waveform.text.strip()
        # Split by comma and convert to float
        data = np.array([float(x) for x in data_str.split(',')])
        # Convert to microvolts
        data *= resolution
        signals.append(data)
    
    return np.stack(signals), leads
lead_names = get_ecg_signal('<UKB resting ECG XML dir>/<subject_id>_20205_2_0.xml')[1]
print(lead_names)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"
i = ngroup-2

# plt.figure()
# sns.heatmap(p_vals[i], cmap='cividis_r')
# print(f'n={n_samples[i]}')
# plt.title(f'$p$-value of T peak time difference (<512ms vs >702ms)')
# plt.xticks([],[])
# plt.yticks([],[])
# plt.savefig('tpeak_ttest_pval.svg')
# plt.figure()
# sns.heatmap(diffs[i], cmap='coolwarm')
# plt.title(f"Cohen's $d$ of T peak time difference (<512ms vs >702ms)")
# plt.xticks([],[])
# plt.yticks([],[])
# plt.savefig('tpeak_ttest_cohend.svg')

plot_combined_heatmap_with_star(p_vals[i], diffs[i], labels=lead_names)
plt.savefig('tpeak_ttest.svg')

In [ ]:

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,      # increase for more global structure (try 30–50)
    min_dist=0.1,        # smaller = tighter clusters
    metric='euclidean',
    verbose=True,
    low_memory=False     # faster if you have RAM
)
toffset = np.stack(ecg_df['Toffset_delay_mat']).reshape(len(ecg_df['Toffset_delay_mat']), -1)
valid_mask = ~np.isnan(toffset).any(-1)
toffset = toffset[valid_mask]
toffset_embedding = reducer.fit_transform(toffset)  # shape: (45000, 2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr, spearmanr

plt.figure(figsize=(15,15))
plot_df = pd.DataFrame({
    'UMAP1': toffset_embedding[:, 0],
    'UMAP2': toffset_embedding[:, 1],
    'EndSystole': np.array(ecg_df['T_ED'])[valid_mask],
})
plot_df["EndSystole_bin"] = pd.qcut(plot_df["EndSystole"], q=5, duplicates="drop")
sns.scatterplot(data=plot_df, x='UMAP1', y='UMAP2', hue='EndSystole_bin', s=10, alpha=0.9, palette='tab10')

corr, p_value = pearsonr(np.array(plot_df["EndSystole_bin"].astype("category").cat.codes), toffset_embedding[:, 0])
print(corr, p_value)
corr, p_value = pearsonr(np.array(plot_df["EndSystole_bin"].astype("category").cat.codes), toffset_embedding[:, 1])
print(corr, p_value)

In [ ]:
ecg_df['Toffset_delay_mat'] = np.stack(ecg_df['Toffset_delay_mat'])

In [ ]:
group_bin = pd.qcut(plot_df["EndSystole"][valid_mask], q=2, duplicates="drop").astype("category").cat
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['Toffset_delay_mat'][valid_mask], np.array(group_bin.codes))
print(np.array(group_bin.categories))
plt.figure()
sns.heatmap(p_vals[0], cmap='cividis_r')
plt.title(f'p-value, n={n_samples[0]}')
plt.figure()
sns.heatmap(diffs[0], cmap='coolwarm')
plt.title(f"Cohen's d, n={n_samples[0]}")

In [ ]:
group_bin = pd.qcut(plot_df["EndSystole"][valid_mask], q=8, duplicates="drop").astype("category").cat
pairs, t_stats, p_vals, n_samples, diffs = pairwise_cell_ttests(ecg_df['Toffset_delay_mat'][valid_mask], np.array(group_bin.codes))

ngroup = len(group_bin.categories.to_numpy())
print(np.array(group_bin.categories))

fig, axes = plt.subplots(ngroup, ngroup-1, figsize=(16, 20), sharex=True, sharey=True)
# axes = np.array(axes)#.ravel()

for i, (pair, p_mat, n_sample, diff) in enumerate(zip(pairs, p_vals, n_samples, diffs)):
    # mask = p_mat >= 0.05
    # print(p_mat[~np.isnan(p_mat)].min(), p_mat[~np.isnan(p_mat)].max())
    sns.heatmap(
        p_mat,
        # mask=mask,
        ax=axes[pair[0], pair[1]-1],
        cmap="cividis_r",
        vmax=1,
        vmin=0,
        cbar=False,
    )
    axes[pair[0], pair[1]-1].set_title(f"p-value, n={n_sample[0]} vs n={n_sample[1]}")

    sns.heatmap(
        diff,
        # mask=mask,
        ax=axes[pair[1], pair[0]],
        cmap="coolwarm",
        center=0,
        # vmax=0.3,
        # vmin=-0.3,
        cbar=True,
        square=True
    )
    axes[pair[1], pair[0]].set_title(f"Cohen's d, n={n_sample[0]} vs n={n_sample[1]}")

# hide unused axes
# for j in range(len(axes)):
#     axes[j].axis("off")

plt.tight_layout()
plt.show()
